# RouteWise recorded demonstration without a GPU

**This notebook replays recorded scores from the completed Colab GPU run. It does not
run a model or classify new tickets.** The eight underlying requests are synthetic demo
examples, and their seven-label scores were produced by the actual trained Qwen adapter.

Select **Runtime → Change runtime type → Hardware accelerator → None (CPU)**,
or choose **Connect without GPU**. Run the three cells below in order.
No model download, backup upload, Google Drive mount, or paid compute is required.

You can demonstrate review thresholds, human decisions, and suspected-incident grouping
using the recorded examples. Every exported record marks whether inference was executed.
Performance measurements remain the original development measurements; replay does not
provide new validation evidence. The live-model notebook is included separately.


In [ ]:
#@title 1. Install the notebook controls (CPU only)
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','ipywidgets>=8.1,<9'])
print('CPU replay controls ready. No GPU is used.')


In [ ]:
#@title 2. Load the included code and recorded GPU scores
from pathlib import Path
import json,sys
package_files={'routewise/__init__.py': '"""RouteWise: inspectable ticket routing and suspected-incident replay."""\n', 'routewise/core.py': '"""Pure Python routing policy, review records, and lexical incident grouping.\n\nNo model calls, network access, or simulated model scores live in this module.\n"""\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timedelta, timezone\nimport math\nimport re\nfrom typing import Mapping\n\nLABELS = (\n    "Support general", "Fileservice", "O365", "EOL", "Software",\n    "Active Directory", "Computer-Services",\n)\nSYSTEM_PROMPT = (\n    "You are an IT helpdesk ticket routing assistant. "\n    "Given a support ticket, respond with exactly one of the following categories: "\n    "Support general, Fileservice, O365, EOL, Software, Active Directory, Computer-Services."\n)\n\n\ndef normalize_scores(log_scores: Mapping[str, float], temperature: float = 1.0) -> dict:\n    if set(log_scores) != set(LABELS):\n        raise ValueError("Scores must contain each of the seven labels exactly once.")\n    if not math.isfinite(temperature) or temperature <= 0:\n        raise ValueError("Temperature must be finite and positive.")\n    values = {k: float(v) for k, v in log_scores.items()}\n    if not all(math.isfinite(v) for v in values.values()):\n        raise ValueError("Non-finite model score; send this ticket for review.")\n    top = max(values.values())\n    weights = {k: math.exp((v - top) / temperature) for k, v in values.items()}\n    total = sum(weights.values())\n    return {k: v / total for k, v in weights.items()}\n\n\ndef validate_scores(scores: Mapping[str, float]) -> None:\n    if set(scores) != set(LABELS):\n        raise ValueError("Missing or unknown routing label.")\n    if not all(math.isfinite(v) and 0 <= v <= 1 for v in scores.values()):\n        raise ValueError("Invalid routing score.")\n    if not math.isclose(sum(scores.values()), 1, abs_tol=1e-5):\n        raise ValueError("Routing scores must sum to one.")\n\n\n@dataclass(frozen=True)\nclass Policy:\n    threshold: float = 0.80\n    min_margin: float = 0.20\n    auto_enabled: bool = False\n\n    def __post_init__(self):\n        if not all(math.isfinite(v) and 0 <= v <= 1\n                   for v in (self.threshold, self.min_margin)):\n            raise ValueError("Threshold and margin must be between zero and one.")\n\n    def decide(self, scores: Mapping[str, float]) -> dict:\n        validate_scores(scores)\n        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)\n        label, score = ranked[0]\n        margin = score - ranked[1][1]\n        automatic = self.auto_enabled and score >= self.threshold and margin >= self.min_margin\n        if not self.auto_enabled:\n            reason = "Automatic routing is disabled until you enable the experimental policy."\n        elif score < self.threshold:\n            reason = "Top candidate score is below the review threshold."\n        elif margin < self.min_margin:\n            reason = "The two leading queues are too close."\n        else:\n            reason = "Meets the experimental score and margin thresholds."\n        return {\n            "predicted_queue": label, "top_score": score, "margin": margin,\n            "status": "automatic" if automatic else "needs_review", "reason": reason,\n            "policy": {"threshold": self.threshold, "min_margin": self.min_margin,\n                       "auto_enabled": self.auto_enabled, "validated": False},\n        }\n\n\ndef policy_metrics(records: list[dict], policy: Policy) -> dict:\n    if not records:\n        raise ValueError("No scored examples supplied.")\n    accepted = []\n    for row in records:\n        if row["truth"] not in LABELS:\n            raise ValueError("Evaluation truth must be a routing label.")\n        decision = policy.decide(row["scores"])\n        if decision["status"] == "automatic":\n            accepted.append(decision["predicted_queue"] == row["truth"])\n    n = len(accepted)\n    return {\n        "total": len(records), "automated": n, "review": len(records) - n,\n        "coverage": n / len(records),\n        "automatic_accuracy": sum(accepted) / n if n else None,\n        "automatic_errors": n - sum(accepted),\n    }\n\n\n@dataclass\nclass Desk:\n    tickets: list = field(default_factory=list)\n    reviews: list = field(default_factory=list)\n\n    def add(self, text: str, scores: dict, policy: Policy, timestamp=None, source="manual"):\n        if not text.strip():\n            raise ValueError("Enter a ticket first.")\n        timestamp = timestamp or datetime.now(timezone.utc)\n        if timestamp.tzinfo is None:\n            raise ValueError("Ticket timestamps need a timezone.")\n        decision = policy.decide(scores)\n        record = {\n            "id": f"T{len(self.tickets) + 1:04d}", "text": text,\n            "timestamp": timestamp.isoformat(), "source": source,\n            "scores": dict(scores), **decision,\n            "human_queue": None,\n        }\n        self.tickets.append(record)\n        return record\n\n    def review(self, ticket_id: str, queue: str):\n        if queue not in LABELS:\n            raise ValueError("Choose a valid queue.")\n        ticket = next((x for x in self.tickets if x["id"] == ticket_id), None)\n        if ticket is None:\n            raise ValueError("Ticket not found.")\n        previous = ticket["human_queue"]\n        ticket["human_queue"] = queue\n        ticket["status"] = "reviewed"\n        self.reviews.append({\n            "ticket_id": ticket_id, "queue": queue, "previous_human_queue": previous,\n            "model_queue": ticket["predicted_queue"],\n            "timestamp": datetime.now(timezone.utc).isoformat(),\n        })\n        return ticket\n\n\n_STOPWORDS = set("i my me we our the a an to of and is are it in on for with since please can cannot not has have this all".split())\n_ALIASES = {\n    "outlook": "email", "mailbox": "email", "mailboxes": "email", "emails": "email",\n    "mail": "email", "messages": "email", "message": "email",\n    "sending": "send", "sent": "send", "sends": "send",\n    "failing": "failure", "failed": "failure", "fails": "failure",\n    "stopped": "failure", "broken": "failure", "down": "failure",\n}\n\n\ndef ticket_terms(text: str) -> Counter:\n    words = re.findall(r"[a-z0-9]+", text.lower())\n    return Counter(_ALIASES.get(w, w) for w in words if w not in _STOPWORDS)\n\n\ndef similarity(a: Counter, b: Counter) -> float:\n    denominator = math.sqrt(sum(v*v for v in a.values()) * sum(v*v for v in b.values()))\n    return sum(v * b.get(k, 0) for k, v in a.items()) / denominator if denominator else 0.0\n\n\ndef suspected_incidents(tickets: list[dict], now=None, window_minutes=10,\n                        min_tickets=3, similarity_threshold=0.55) -> list[dict]:\n    """Complete-link grouping avoids merging dissimilar endpoints through a bridge.\n\n    Lexical similarity with documented email aliases; not root-cause diagnosis.\n    Each distinct ticket counts once. Synthetic reporters do not prove real impact.\n    """\n    if window_minutes <= 0 or min_tickets < 2 or not 0 < similarity_threshold <= 1:\n        raise ValueError("Invalid incident grouping settings.")\n    if not tickets:\n        return []\n    times = [datetime.fromisoformat(x["timestamp"]) for x in tickets]\n    if any(x.tzinfo is None for x in times):\n        raise ValueError("Incident timestamps need a timezone.")\n    now = now or max(times)\n    if now.tzinfo is None:\n        raise ValueError("Current time needs a timezone.")\n    recent = {}\n    for ticket, time in zip(tickets, times):\n        if now - timedelta(minutes=window_minutes) <= time <= now:\n            recent[ticket["id"]] = ticket\n    groups = []\n    for ticket in sorted(recent.values(), key=lambda x: (x["timestamp"], x["id"])):\n        terms = ticket_terms(ticket["text"])\n        match = next((g for g in groups if all(\n            similarity(terms, ticket_terms(other["text"])) >= similarity_threshold\n            for other in g)), None)\n        if match is None:\n            groups.append([ticket])\n        else:\n            match.append(ticket)\n    return [{\n        "title": "Suspected shared incident", "ticket_ids": [x["id"] for x in group],\n        "count": len(group), "example": group[0]["text"],\n        "method": "Lexical similarity with email aliases; complete-link grouping",\n        "confirmed": False,\n    } for group in groups if len(group) >= min_tickets]\n', 'routewise/demo.py': '"""A notebook-native desk. No public server or external ticket integrations."""\nfrom datetime import datetime, timedelta, timezone\nfrom html import escape\nfrom pathlib import Path\nimport json\nimport ipywidgets as w\nfrom IPython.display import display, HTML, clear_output\nfrom .core import Desk, Policy, LABELS, suspected_incidents\n\nSCENARIOS = [\n    ("Please install the approved Adobe Acrobat application on my laptop.", "Software"),\n    ("I cannot access the shared folder through Teams. Please investigate.", None),\n    ("Outlook email sending failed for our team this morning.", "O365"),\n    ("Our team email sending failed in Outlook this morning.", "O365"),\n    ("Outlook email sending has failed again for our team this morning.", "O365"),\n    ("Please decommission the retired test server and remove it from monitoring.", "EOL"),\n    ("Please create an Active Directory account for the new employee.", "Active Directory"),\n    ("The office printer has a persistent paper jam and needs repair.", "Computer-Services"),\n]\n\n\ndef launch_desk(router):\n    desk = Desk()\n    recorded_mode = bool(getattr(router, \'recorded\', False))\n    output = w.Output()\n    text = w.Textarea(placeholder="Describe the support request…", layout=w.Layout(width="100%", height="90px"))\n    threshold = w.FloatSlider(value=.80, min=.15, max=.99, step=.01,\n                              description="Score cutoff", style={"description_width": "initial"})\n    margin = w.FloatSlider(value=.20, min=0, max=.8, step=.01,\n                           description="Top-two gap", style={"description_width": "initial"})\n    automatic = w.Checkbox(value=False, description="Enable experimental automatic routing")\n    route = w.Button(description="Route ticket", button_style="primary")\n    replay = w.Button(description="Replay 8 recorded tickets" if recorded_mode else "Replay 8 synthetic tickets")\n    refresh = w.Button(description="Refresh desk")\n    export = w.Button(description="Download desk records")\n    ticket_select = w.Dropdown(options=[], description="Ticket")\n    queue_select = w.Dropdown(options=LABELS, description="Human queue")\n    review = w.Button(description="Save human decision")\n    notice = w.HTML()\n\n    def policy():\n        return Policy(threshold.value, margin.value, automatic.value)\n\n    def table(headers, rows):\n        return "<table style=\'width:100%;border-collapse:collapse\'>" + "<tr>" + "".join(\n            f"<th style=\'text-align:left;padding:8px;border-bottom:1px solid #aaa\'>{escape(str(h))}</th>"\n            for h in headers) + "</tr>" + "".join("<tr>" + "".join(\n                f"<td style=\'padding:8px;border-bottom:1px solid #ddd\'>{escape(str(v))}</td>"\n                for v in row) + "</tr>" for row in rows) + "</table>"\n\n    def render():\n        ticket_select.options = [(f"{r[\'id\']} · {r[\'status\']}", r[\'id\']) for r in desk.tickets]\n        with output:\n            clear_output(wait=True)\n            auto = sum(r["status"] == "automatic" for r in desk.tickets)\n            pending = sum(r["status"] == "needs_review" for r in desk.tickets)\n            display(HTML(f"<h3>Ticket desk</h3><p>{len(desk.tickets)} tickets · {auto} automatic · "\n                         f"{pending} awaiting review · {len(desk.reviews)} human decisions</p>"))\n            display(HTML("<p>Scores are relative preferences across seven labels, not validated probabilities "\n                         "of correctness. Threshold changes affect new decisions only.</p>"))\n            if recorded_mode:\n                display(HTML("<p><b>Recorded replay:</b> these are saved scores from the completed GPU run. "\n                             "No model inference is executed in this notebook. Only the eight recorded texts are available.</p>"))\n            if desk.tickets:\n                display(HTML(table(["ID", "Ticket", "Suggested queue", "Status", "Human queue", "Source"], [\n                    [r[\'id\'], r[\'text\'], r[\'predicted_queue\'] or \'Inference failed\', r[\'status\'],\n                     r[\'human_queue\'] or \'—\', r[\'source\']] for r in desk.tickets])))\n                latest = desk.tickets[-1]\n                if latest[\'scores\']:\n                    display(HTML("<h3>Latest ticket candidate scores</h3>" + table(\n                        ["Queue", "Relative score"],\n                        [(k, f"{v:.3f}") for k, v in sorted(latest[\'scores\'].items(), key=lambda x:x[1], reverse=True)])))\n                display(HTML(f"<p>{escape(latest[\'reason\'])}</p>"))\n                incidents = suspected_incidents(desk.tickets)\n                display(HTML("<h3>Incident Radar</h3><p>Lexical grouping over a 10-minute window; "\n                             "three reports required. Groups are suspected, not confirmed outages.</p>"))\n                for group in incidents:\n                    display(HTML(f"<p><b>{escape(group[\'title\'])}</b>: {group[\'count\']} reports · "\n                                 f"{escape(\', \'.join(group[\'ticket_ids\']))}<br>{escape(group[\'example\'])}</p>"))\n                if not incidents:\n                    display(HTML("<p>No group currently meets the incident settings.</p>"))\n\n    def add(ticket, timestamp=None, source="manual"):\n        try:\n            result = router.score(ticket)\n            record = desk.add(ticket, result[\'scores\'], policy(), timestamp, source)\n            record.update({k: result[k] for k in [\'log_scores\', \'latency_ms\', \'score_method\', \'calibrated\']})\n            record[\'inference_mode\'] = \'recorded_scores\' if recorded_mode else \'live_model\'\n            record[\'new_inference_executed\'] = not recorded_mode\n        except Exception as error:\n            # Preserve inference failures as review items, with no invented label or score.\n            record = {\n                \'id\': f\'T{len(desk.tickets)+1:04d}\', \'text\': ticket,\n                \'timestamp\': (timestamp or datetime.now(timezone.utc)).isoformat(),\n                \'source\': source, \'scores\': {}, \'predicted_queue\': None,\n                \'human_queue\': None, \'status\': \'needs_review\',\n                \'reason\': f\'Inference failed: {type(error).__name__}: {error}\',\n            }\n            desk.tickets.append(record)\n        return record\n\n    def on_route(_):\n        if not text.value.strip():\n            notice.value = "Enter a ticket first."\n            return\n        route.disabled = replay.disabled = True\n        notice.value = "Reading saved GPU scores…" if recorded_mode else "Scoring all seven complete labels…"\n        try:\n            add(text.value)\n            render()\n            notice.value = "Ticket processed."\n        finally:\n            route.disabled = replay.disabled = False\n\n    def on_replay(_):\n        route.disabled = replay.disabled = True\n        start = datetime.now(timezone.utc) - timedelta(minutes=4)\n        try:\n            for i, (ticket, expected) in enumerate(SCENARIOS):\n                notice.value = f"Replaying saved ticket {i+1}/{len(SCENARIOS)}…" if recorded_mode else f"Scoring synthetic ticket {i+1}/{len(SCENARIOS)}…"\n                record = add(ticket, start + timedelta(seconds=30*i),\n                             \'recorded_synthetic_replay\' if recorded_mode else \'synthetic_replay\')\n                record[\'illustrative_expected_queue\'] = expected\n                render()\n            notice.value = "Synthetic replay complete. These examples are not a benchmark."\n        finally:\n            route.disabled = replay.disabled = False\n\n    def on_review(_):\n        if ticket_select.value:\n            desk.review(ticket_select.value, queue_select.value)\n            render()\n            notice.value = "Human decision saved. Model weights have not changed."\n\n    def on_export(_):\n        path = Path(\'/content/routewise_desk_records.json\')\n        path.write_text(json.dumps({\n            \'tickets\': desk.tickets, \'reviews\': desk.reviews,\n            \'notice\': \'Experimental policy; synthetic replay is not benchmark evidence.\',\n        }, indent=2), encoding=\'utf-8\')\n        from google.colab import files\n        files.download(str(path))\n\n    route.on_click(on_route)\n    replay.on_click(on_replay)\n    refresh.on_click(lambda _: render())\n    review.on_click(on_review)\n    export.on_click(on_export)\n    display(w.VBox([\n        w.HTML("<h2>RouteWise Incident Desk</h2><p>" +\n               ("Recorded GPU scores · no live model inference" if recorded_mode else "Your restored Qwen LoRA model") +\n               " · experimental review policy · notebook session storage</p>"),\n        text, w.HBox([threshold, margin]), automatic,\n        w.HBox([route, replay, refresh, export]), notice,\n        w.HBox([ticket_select, queue_select, review]), output,\n    ]))\n    render()\n    return desk\n', 'routewise/recorded.py': '"""Offline access to recorded GPU scores. This class never runs a model."""\nfrom copy import deepcopy\nfrom .core import validate_scores\n\n\nclass RecordedRouter:\n    recorded = True\n\n    def __init__(self, tickets):\n        self.records = {}\n        for ticket in tickets:\n            validate_scores(ticket[\'scores\'])\n            key = ticket[\'text\'].strip()\n            if key in self.records:\n                raise ValueError(\'Duplicate recorded text; choose a specific run before replaying.\')\n            self.records[key] = deepcopy(ticket)\n\n    def score(self, text):\n        key = text.strip()\n        if key not in self.records:\n            raise ValueError(\'This text was not recorded. New tickets require the live GPU notebook.\')\n        row = self.records[key]\n        return {\n            \'prediction\':max(row[\'scores\'],key=row[\'scores\'].get),\n            \'scores\':deepcopy(row[\'scores\']), \'log_scores\':deepcopy(row[\'log_scores\']),\n            \'latency_ms\':row[\'latency_ms\'],\n            \'score_method\':row[\'score_method\'], \'calibrated\':False,\n            \'inference_mode\':\'recorded_scores\', \'new_inference_executed\':False,\n            \'latency_is_historical\':True,\n        }\n'}
for relative,source in package_files.items():
    target=Path("/content")/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(source,encoding="utf-8")
if "/content" not in sys.path: sys.path.insert(0,"/content")
recorded_export={'tickets': [{'id': 'T0001', 'text': 'Please install the approved Adobe Acrobat application on my laptop.', 'timestamp': '2026-11-09T01:00:00.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.06375793597384695, 'Fileservice': 0.06371161992364203, 'O365': 0.00270752026314559, 'EOL': 5.287152354102382e-05, 'Software': 0.8615345373969809, 'Active Directory': 0.0037350169381758164, 'Computer-Services': 0.004500497980667584}, 'predicted_queue': 'Software', 'top_score': 0.8615345373969809, 'margin': 0.797776601423134, 'status': 'needs_review', 'reason': 'Top candidate score is below the review threshold.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': None, 'log_scores': {'Support general': -10.939176559448242, 'Fileservice': -10.939903259277344, 'O365': -14.098237037658691, 'EOL': -18.034160614013672, 'Software': -8.335555076599121, 'Active Directory': -13.776517868041992, 'Computer-Services': -13.590082168579102}, 'latency_ms': 688.7188169998808, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': 'Software'}, {'id': 'T0002', 'text': 'I cannot access the shared folder through Teams. Please investigate.', 'timestamp': '2026-11-09T01:00:30.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.2881272913624103, 'Fileservice': 0.36744725345109286, 'O365': 0.15727270925076128, 'EOL': 0.0019159398436619055, 'Software': 0.03177321677498057, 'Active Directory': 0.06666284241479103, 'Computer-Services': 0.08680074690230205}, 'predicted_queue': 'Fileservice', 'top_score': 0.36744725345109286, 'margin': 0.07931996208868258, 'status': 'needs_review', 'reason': 'Top candidate score is below the review threshold.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': None, 'log_scores': {'Support general': -9.41989517211914, 'Fileservice': -9.176717758178711, 'O365': -10.02531623840332, 'EOL': -14.433089256286621, 'Software': -11.624673843383789, 'Active Directory': -10.883649826049805, 'Computer-Services': -10.619682312011719}, 'latency_ms': 654.0621160002047, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': None}, {'id': 'T0003', 'text': 'Outlook email sending failed for our team this morning.', 'timestamp': '2026-11-09T01:01:00.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.019458637311413004, 'Fileservice': 0.00018056190438557002, 'O365': 0.9663963240739645, 'EOL': 0.00021919868506597496, 'Software': 0.0036126987380642995, 'Active Directory': 0.010121228434889959, 'Computer-Services': 1.1350852216794451e-05}, 'predicted_queue': 'O365', 'top_score': 0.9663963240739645, 'margin': 0.9469376867625515, 'status': 'automatic', 'reason': 'Meets the experimental score and margin thresholds.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': None, 'log_scores': {'Support general': -12.119866371154785, 'Fileservice': -16.79983901977539, 'O365': -8.214583396911621, 'EOL': -16.605934143066406, 'Software': -13.803702354431152, 'Active Directory': -12.77352237701416, 'Computer-Services': -19.566619873046875}, 'latency_ms': 657.4595669999326, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': 'O365'}, {'id': 'T0004', 'text': 'Our team email sending failed in Outlook this morning.', 'timestamp': '2026-11-09T01:01:30.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.12900403390589693, 'Fileservice': 0.0011720538417385688, 'O365': 0.8209850930574468, 'EOL': 0.0008753691236198971, 'Software': 0.010204800809124717, 'Active Directory': 0.037601615144904066, 'Computer-Services': 0.00015703411726885951}, 'predicted_queue': 'O365', 'top_score': 0.8209850930574468, 'margin': 0.6919810591515498, 'status': 'needs_review', 'reason': 'Top candidate score is below the review threshold.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': None, 'log_scores': {'Support general': -10.228742599487305, 'Fileservice': -14.929828643798828, 'O365': -8.378081321716309, 'EOL': -15.221695899963379, 'Software': -12.765727996826172, 'Active Directory': -11.461539268493652, 'Computer-Services': -16.939878463745117}, 'latency_ms': 667.4211590002415, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': 'O365'}, {'id': 'T0005', 'text': 'Outlook email sending has failed again for our team this morning.', 'timestamp': '2026-11-09T01:02:00.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.01534666490790745, 'Fileservice': 0.0002666739763038089, 'O365': 0.9719693164187311, 'EOL': 0.00014048630706644708, 'Software': 0.0037661847244192825, 'Active Directory': 0.008501656046778137, 'Computer-Services': 9.017618793805838e-06}, 'predicted_queue': 'O365', 'top_score': 0.9719693164187311, 'margin': 0.9566226515108236, 'status': 'automatic', 'reason': 'Meets the experimental score and margin thresholds.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': None, 'log_scores': {'Support general': -12.351611137390137, 'Fileservice': -16.404237747192383, 'O365': -8.203185081481934, 'EOL': -17.045154571533203, 'Software': -13.756446838378906, 'Active Directory': -12.942248344421387, 'Computer-Services': -19.79108428955078}, 'latency_ms': 506.32836499971745, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': 'O365'}, {'id': 'T0006', 'text': 'Please decommission the retired test server and remove it from monitoring.', 'timestamp': '2026-11-09T01:02:30.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.008059711689210543, 'Fileservice': 4.078884752451162e-05, 'O365': 0.000641016984853485, 'EOL': 0.9878940802832508, 'Software': 0.0012725879589452005, 'Active Directory': 0.0013132422072686533, 'Computer-Services': 0.0007785720289468321}, 'predicted_queue': 'EOL', 'top_score': 0.9878940802832508, 'margin': 0.9798343685940403, 'status': 'automatic', 'reason': 'Meets the experimental score and margin thresholds.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': None, 'log_scores': {'Support general': -13.039960861206055, 'Fileservice': -18.32618522644043, 'O365': -15.571537971496582, 'EOL': -8.231263160705566, 'Software': -14.885786056518555, 'Active Directory': -14.854339599609375, 'Computer-Services': -15.377132415771484}, 'latency_ms': 500.6503159997919, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': 'EOL'}, {'id': 'T0007', 'text': 'Please create an Active Directory account for the new employee.', 'timestamp': '2026-11-09T01:03:00.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.0012959785212568477, 'Fileservice': 1.4406648902620425e-05, 'O365': 3.916207887123498e-05, 'EOL': 1.4499494395775407e-05, 'Software': 0.000612714838538722, 'Active Directory': 0.9976088848662242, 'Computer-Services': 0.00041435355181049965}, 'predicted_queue': 'Active Directory', 'top_score': 0.9976088848662242, 'margin': 0.9963129063449674, 'status': 'automatic', 'reason': 'Meets the experimental score and margin thresholds.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': None, 'log_scores': {'Support general': -14.862380027770996, 'Fileservice': -19.361711502075195, 'O365': -18.361692428588867, 'EOL': -19.355287551879883, 'Software': -15.611501693725586, 'Active Directory': -8.21628475189209, 'Computer-Services': -16.002681732177734}, 'latency_ms': 497.7192860001196, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': 'Active Directory'}, {'id': 'T0008', 'text': 'The office printer has a persistent paper jam and needs repair.', 'timestamp': '2026-11-09T01:03:30.321582+00:00', 'source': 'synthetic_replay', 'scores': {'Support general': 0.170153098842721, 'Fileservice': 0.007625165114824543, 'O365': 0.0005502336795844291, 'EOL': 8.673131852192999e-06, 'Software': 0.0031071620885862045, 'Active Directory': 0.00561609382097088, 'Computer-Services': 0.8129395733214607}, 'predicted_queue': 'Computer-Services', 'top_score': 0.8129395733214607, 'margin': 0.6427864744787397, 'status': 'reviewed', 'reason': 'Top candidate score is below the review threshold.', 'policy': {'threshold': 0.9, 'min_margin': 0.2, 'auto_enabled': True, 'validated': False}, 'human_queue': 'Computer-Services', 'log_scores': {'Support general': -9.947500228881836, 'Fileservice': -13.05274486541748, 'O365': -15.681611061096191, 'EOL': -19.831724166870117, 'Software': -13.950489044189453, 'Active Directory': -13.358562469482422, 'Computer-Services': -8.38354206085205}, 'latency_ms': 481.3143310002488, 'score_method': 'Sum of complete-label and EOS log likelihoods', 'calibrated': False, 'illustrative_expected_queue': 'Computer-Services'}], 'reviews': [{'ticket_id': 'T0008', 'queue': 'Computer-Services', 'previous_human_queue': None, 'model_queue': 'Computer-Services', 'timestamp': '2026-11-09T01:06:27.965427+00:00'}], 'notice': 'Experimental policy; synthetic replay is not benchmark evidence.'}
from routewise.recorded import RecordedRouter
recorded_router=RecordedRouter(recorded_export["tickets"])
print("Loaded",len(recorded_export["tickets"]),"recorded tickets. No inference executed.")
print("Original export SHA256:",'e47318c69b71d111f594609e0f0fdb22344b20fe70b9cd313ee145bf21e644b5')


In [ ]:
#@title 3. Open the recorded replay desk
from routewise.demo import launch_desk
desk=launch_desk(recorded_router)


## Demonstration sequence

1. Set **Score cutoff** to **0.90**, keep **Top-two gap** at **0.20**, and enable
   the experimental policy.
2. Click **Replay 8 recorded tickets** once. Expect four automatic routes and four
   requests for review, using the original recorded scores.
3. Choose **T0008**, select **Computer-Services**, and save the human decision.
   Expect four automatic, three awaiting review, and one human decision.
4. Scroll to **Incident Radar**: T0003, T0004, and T0005 form a suspected group.
5. Download desk records before closing the runtime. They explicitly mark recorded mode.

The editable text box accepts only exact texts from the eight-record collection.
Other text becomes an explicit review item with no invented score. Use the live notebook
when GPU access is available for new-ticket inference. Model weights do not change when
a reviewer confirms a queue. Incident grouping is lexical, not a root-cause diagnosis.
